# Entity Binding — task demo

Three sections:

1. **Causal model** — entities, query/answer structure, the positional variables that make this task analyzable.
2. **Templates & token positions** — the per-(group, slot) statement positions, located via prefix tokenization.
3. **Counterfactual generators** — `swap_query_group` (the task's signature CF), `random_counterfactual`, and the loader-convention `generate_dataset` (query_group-only resampling).

Tokenization uses `gpt2`. We use the default "love" config (2 groups × 2 slots: person + food). No interventions — see `analyses/locate/demo.ipynb` for those.

In [1]:
from causalab.tasks.entity_binding.config import create_sample_love_config
from causalab.tasks.entity_binding.causal_models import (
    create_positional_entity_causal_model,
    sample_valid_entity_binding_input,
)
from causalab.tasks.entity_binding.counterfactuals import (
    swap_query_group,
    random_counterfactual,
    generate_dataset,
)
from causalab.tasks.entity_binding.token_positions import create_token_positions

config = create_sample_love_config()
model = create_positional_entity_causal_model(config)

(model.id, config.max_groups, config.max_entities_per_group, config.entity_roles, config.question_templates)

('entity_binding_positional_entity_2g_2e',
 2,
 2,
 {0: 'person', 1: 'food'},
 {((0,), 1): 'What does {person} love?', ((1,), 0): 'Who loves {food}?'})

## 1. Causal model

Inputs include `entity_g{g}_e{e}` for each (group, slot), `query_group`, `query_indices`, `answer_index`. Computed variables include `query_e{e}`, `positional_query_e{e}`, `positional_answer`, `raw_input`, `raw_output`. `positional_answer` is the analytical target — the single integer the model must compute internally.

In [2]:
import random

random.seed(0)
trace = sample_valid_entity_binding_input(config, model=model)

print("Inputs:")
print(f"  query_group   = {trace['query_group']}    (asking about group {trace['query_group']})")
print(f"  query_indices = {trace['query_indices']}    (given slot index {trace['query_indices'][0]} = {config.entity_roles[trace['query_indices'][0]]})")
print(f"  answer_index  = {trace['answer_index']}    (asking for slot index {trace['answer_index']} = {config.entity_roles[trace['answer_index']]})")
for g in range(config.max_groups):
    parts = [f"e{e}={trace[f'entity_g{g}_e{e}']!r}" for e in range(config.max_entities_per_group)]
    print(f"  group {g}: " + ", ".join(parts))
print()
print("Computed:")
for e in range(config.max_entities_per_group):
    print(f"  query_e{e}            = {trace[f'query_e{e}']!r}")
for e in range(config.max_entities_per_group):
    print(f"  positional_query_e{e} = {trace[f'positional_query_e{e}']}")
print(f"  positional_answer    = {trace['positional_answer']}")
print(f"  raw_output           = {trace['raw_output']!r}")
print()
print("raw_input:")
print(trace['raw_input'])

Inputs:
  query_group   = 1    (asking about group 1)
  query_indices = (1,)    (given slot index 1 = food)
  answer_index  = 0    (asking for slot index 0 = person)
  group 0: e0='Tim', e1='soup'
  group 1: e0='Sue', e1='bread'

Computed:
  query_e0            = 'Sue'
  query_e1            = 'bread'
  positional_query_e0 = ()
  positional_query_e1 = (1,)
  positional_answer    = 1
  raw_output           = 'Sue'

raw_input:
We will ask a question about the following sentences.

Tim loves soup and Sue loves bread. Who loves bread?
Answer: 


Note how the positional variables decompose retrieval:

- `query_e{e}` recovers the entity *of the query group* at slot `e` — i.e. what the question is asking about.
- `positional_query_e{e}` is the (singleton) set of group indices where the query entity appears at slot `e`.
- `positional_answer` is the intersection — the single group from which to retrieve. **This is the variable analyses target.**

## 2. Templates & token positions

`create_token_positions(pipeline, template, config)` returns a `last` position plus `g{g}_e{e}_last` positions for every (group, slot) — the per-entity factories run prefix tokenization to find each entity's last token in the **statement region** (not the question, where the same entity often re-appears). The demo below shows the `last` position only; the per-entity positions resolve correctly when consumed by the causalab analysis layer (which feeds them through the same padded pipeline used to extract activations).

In [3]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=4, max_length=128)
template_for_positions = config.build_mega_template(
    config.max_groups, trace["query_indices"], trace["answer_index"]
)
positions = create_token_positions(pipeline, template_for_positions, config)
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['last', 'g0_e0_last', 'g0_e1_last', 'g1_e0_last', 'g1_e1_last']

In [4]:
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id
non_pad = [i for i, t in enumerate(ids) if t != pad_id]

last_idx = positions["last"].index(trace)[0]

print(f"`last` resolves to padded-token index {last_idx}: {decoded[last_idx]!r}")
print()
print("Last 12 non-pad tokens of the prompt:")
print(f"{'idx':>4}  {'token':<14}  marker")
for i in non_pad[-12:]:
    marker = "<-- last" if i == last_idx else ""
    print(f"{i:>4}  {decoded[i]!r:<14}  {marker}")

`last` resolves to padded-token index 127: ' '

Last 12 non-pad tokens of the prompt:
 idx  token           marker
 116  ' Sue'          
 117  ' loves'        
 118  ' bread'        
 119  '.'             
 120  ' Who'          
 121  ' loves'        
 122  ' bread'        
 123  '?'             
 124  '\n'            
 125  'Answer'        
 126  ':'             
 127  ' '             <-- last


## 3. Counterfactual generators

Three generators worth comparing. Each produces an `{input, counterfactual_inputs}` pair; we render the entity layout and the key computed variables to see what changes.

In [5]:
def show_pair(label, ex):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- {label} ---")
    for tag, t in [("base", base), ("cf  ", cf)]:
        groups = []
        for g in range(config.max_groups):
            parts = [t[f'entity_g{g}_e{e}'] for e in range(config.max_entities_per_group)]
            groups.append(f"g{g}=({','.join(parts)})")
        print(f"  {tag}: " + "  ".join(groups) +
              f"  query_group={t['query_group']}  query_e0={t['query_e0']!r}  "
              f"positional_answer={t['positional_answer']}  raw_output={t['raw_output']!r}")
    print()

In [6]:
random.seed(1)
show_pair("swap_query_group", swap_query_group(config))
show_pair("random_counterfactual", random_counterfactual(config))
show_pair("generate_dataset (query_group resample)", generate_dataset(model, n=1, seed=1)[0])

--- swap_query_group ---
  base: g0=(Pete,bread)  g1=(Sue,soup)  query_group=0  query_e0='Pete'  positional_answer=0  raw_output='bread'
  cf  : g0=(Sue,soup)  g1=(Pete,bread)  query_group=1  query_e0='Pete'  positional_answer=1  raw_output='bread'

--- random_counterfactual ---
  base: g0=(Pete,bread)  g1=(Sue,tea)  query_group=0  query_e0='Pete'  positional_answer=0  raw_output='bread'
  cf  : g0=(Pete,jam)  g1=(Ann,tea)  query_group=0  query_e0='Pete'  positional_answer=0  raw_output='jam'

--- generate_dataset (query_group resample) ---
  base: g0=(Pete,bread)  g1=(Sue,soup)  query_group=0  query_e0='Pete'  positional_answer=0  raw_output='bread'
  cf  : g0=(Pete,bread)  g1=(Sue,soup)  query_group=1  query_e0='Sue'  positional_answer=1  raw_output='soup'



Reading the output:

- **`swap_query_group`** swaps groups *and* updates `query_group` so the question still references the same entity. Entity layout differs between base and CF; the question text is identical; `positional_answer` flips. Use this when you want to test whether the model has learned positional binding (`positional_answer`) versus just identity matching.
- **`random_counterfactual`** draws two independent samples — every variable may differ. Useful as a distribution baseline.
- **`generate_dataset`** keeps every entity fixed and only resamples `query_group`. Both `positional_answer` and `raw_output` change cleanly. This is the loader convention's choice for `analysis/locate` in pairwise mode (`docs/CODEBASE.md` §7).

## Next steps

Compose a runner config that mounts `task: entity_binding` and the analyses you need (e.g. `analysis/baseline`, `analysis/locate`). The natural research questions are:

- Where is `positional_answer` computed? (locate, subspace)
- Does the subspace that encodes it preserve the binding-vs-identity distinction? (path_steering with `swap_query_group` CFs)

Outputs land under `artifacts/entity_binding/<model>/<analysis>/...`.